In [14]:
import numpy as np

TASK 0 — Data Loading and Preprocessing

In [ ]:
DELIVERIES = "./deliveries.csv"
MATCHES    = "./matches.csv"
 

del_raw = np.genfromtxt(
    DELIVERIES,
    delimiter=",",
    dtype=str,
    skip_header=1,
    filling_values="",
    encoding="utf-8",
    autostrip=True,
)
 

match_id      = del_raw[:, 0].astype(int)
batting_team  = del_raw[:, 2]
bowling_team  = del_raw[:, 3]
over          = del_raw[:, 4].astype(int)   
ball          = del_raw[:, 5].astype(int)
batter        = del_raw[:, 6]
bowler        = del_raw[:, 7]
batsman_runs  = del_raw[:, 9].astype(int)
extra_runs    = del_raw[:, 10].astype(int)
total_runs    = del_raw[:, 11].astype(int)
is_wicket     = del_raw[:, 13].astype(int)

print(f"Deliveries loaded : {len(match_id):,} rows")
print(f"Sample batter     : {batter[:3]}")
print(f"Sample match_ids  : {match_id[:3]}")
 

 
import csv as _csv
 
with open(MATCHES, encoding="utf-8", newline="") as fh:
    reader     = _csv.reader(fh)
    _header    = next(reader)           
    mat_rows   = [row for row in reader]
 
mat_raw = np.array(mat_rows, dtype=str)
 
m_id          = mat_raw[:, 0].astype(int)
m_toss_winner = np.char.strip(mat_raw[:, 9])
m_winner      = np.char.strip(mat_raw[:, 11])
 
print(f"Matches loaded    : {len(m_id):,} rows")
print()

Deliveries loaded : 260,920 rows
Sample batter     : ['SC Ganguly' 'BB McCullum' 'BB McCullum']
Sample match_ids  : [335982 335982 335982]
Matches loaded    : 1,095 rows



TASK 1 — Total Runs per Match

In [27]:
unique_matches = np.unique(match_id)                    

total_runs_per_match = np.array(
    [total_runs[match_id == mid].sum() for mid in unique_matches]
)
 
top10_idx = np.argsort(total_runs_per_match)[::-1][:10]
print(f"{'Match ID':>10}  {'Total Runs':>12}")

print("-" * 26)
for i in top10_idx:
    print(f"{unique_matches[i]:>10}  {total_runs_per_match[i]:>12}")
print()

  Match ID    Total Runs
--------------------------
   1426268           549
   1426280           523
   1422126           523
   1426281           504
    419137           469
   1426273           465
   1136604           459
   1359512           458
   1082641           453
   1216527           449



TASK 2 — Top 5 Batters

In [17]:
unique_batters = np.unique(batter)
batter_totals  = np.array(
    [batsman_runs[batter == b].sum() for b in unique_batters]
)
 
top5_idx = np.argsort(batter_totals)[::-1][:5]
print(f"{'Rank':<5} {'Player':<30} {'Total Runs':>10}")
print("-" * 48)
for rank, i in enumerate(top5_idx, 1):
    print(f"{rank:<5} {unique_batters[i]:<30} {batter_totals[i]:>10}")
print()
 

Rank  Player                         Total Runs
------------------------------------------------
1     V Kohli                              8014
2     S Dhawan                             6769
3     RG Sharma                            6630
4     DA Warner                            6567
5     SK Raina                             5536



TASK 3 — Strike Rate (top 10 by total runs, min 100 balls)

In [18]:
balls_faced = np.array(
    [(batter == b).sum() for b in unique_batters]
)
 
min_balls = balls_faced >= 100
q_batters = unique_batters[min_balls]
q_runs    = batter_totals[min_balls]
q_balls   = balls_faced[min_balls]
 
strike_rate = (q_runs / q_balls) * 100
 
sr_top10 = np.argsort(strike_rate)[::-1][:10]
print(f"{'Player':<30} {'Runs':>6} {'Balls':>6} {'SR':>8}")
print("-" * 55)
for i in sr_top10:
    print(f"{q_batters[i]:<30} {q_runs[i]:>6} {q_balls[i]:>6} {strike_rate[i]:>8.2f}")
print()

Player                           Runs  Balls       SR
-------------------------------------------------------
J Fraser-McGurk                   330    150   220.00
WG Jacks                          230    133   172.93
PD Salt                           653    385   169.61
T Stubbs                          405    239   169.46
TM Head                           772    458   168.56
AD Russell                       2488   1515   164.22
BCJ Cutting                       238    146   163.01
H Klaasen                         993    613   161.99
Ramandeep Singh                   170    106   160.38
Ashutosh Sharma                   189    118   160.17



TASK 4 — Bowler Economy Rate (top 10 economical, min 60 balls)

In [19]:

unique_bowlers = np.unique(bowler)
 
bowler_runs_conceded = np.array(
    [total_runs[bowler == bw].sum() for bw in unique_bowlers]
)
bowler_balls = np.array(
    [(bowler == bw).sum() for bw in unique_bowlers]
)
 

min_b = bowler_balls >= 60
e_bowlers = unique_bowlers[min_b]
e_runs    = bowler_runs_conceded[min_b]
e_balls   = bowler_balls[min_b]
 
overs_bowled = e_balls / 6.0
economy      = e_runs / overs_bowled
 
econ_top10 = np.argsort(economy)[:10]   # lowest economy = best
print(f"{'Bowler':<28} {'Runs':>6} {'Overs':>8} {'Economy':>9}")
print("-" * 55)
for i in econ_top10:
    print(f"{e_bowlers[i]:<28} {e_runs[i]:>6} {overs_bowled[i]:>8.1f} {economy[i]:>9.2f}")
print()
 

Bowler                         Runs    Overs   Economy
-------------------------------------------------------
Sohail Tanvir                   275     44.2      6.23
A Chandila                      245     39.0      6.28
FH Edwards                      160     25.0      6.40
JW Hastings                      66     10.2      6.49
SMSM Senanayake                 211     32.5      6.49
MJ Clarke                        72     11.0      6.55
SM Pollock                      307     46.7      6.58
SM Harwood                       74     11.2      6.63
A Kumble                       1089    163.8      6.65
GD McGrath                      366     54.8      6.67



TASK 5 — Average Runs per Over (overs 1–20)

In [20]:
over_numbers = np.arange(0, 20)
 

over_total_runs = np.array(
    [batsman_runs[over == ov].sum() for ov in over_numbers]
)
 

over_match_count = np.array(
    [np.unique(match_id[over == ov]).size for ov in over_numbers]
)
 
avg_runs_per_over = over_total_runs / over_match_count
 
print(f"{'Over':>5} {'Avg Runs':>10}")
print("-" * 18)
for ov, avg in zip(over_numbers + 1, avg_runs_per_over):
    bar = "█" * int(avg)
    print(f"{ov:>5} {avg:>10.2f}  {bar}")
print()

 Over   Avg Runs
------------------
    1      11.32  ███████████
    2      13.60  █████████████
    3      15.46  ███████████████
    4      15.99  ███████████████
    5      16.21  ████████████████
    6      16.13  ████████████████
    7      12.83  ████████████
    8      13.95  █████████████
    9      14.56  ██████████████
   10      14.31  ██████████████
   11      14.77  ██████████████
   12      15.07  ███████████████
   13      15.10  ███████████████
   14      15.58  ███████████████
   15      16.01  ████████████████
   16      16.25  ████████████████
   17      16.77  ████████████████
   18      17.28  █████████████████
   19      17.00  ████████████████
   20      15.95  ███████████████



TASK 6 — Boundary Analysis

In [21]:
fours = np.sum(batsman_runs == 4)
sixes = np.sum(batsman_runs == 6)
print(f"Total Fours : {fours:,}")
print(f"Total Sixes : {sixes:,}")
 
# Bonus — which batting team hit most boundaries?
boundary_mask = (batsman_runs == 4) | (batsman_runs == 6)
unique_teams  = np.unique(batting_team)
team_boundaries = np.array(
    [np.sum(boundary_mask & (batting_team == t)) for t in unique_teams]
)
 
top_team_idx = np.argmax(team_boundaries)
print(f"\nTeam with most boundaries : {unique_teams[top_team_idx]}"
      f" ({team_boundaries[top_team_idx]:,} boundaries)")
 
# Top 5 teams
team_top5 = np.argsort(team_boundaries)[::-1][:5]
print(f"\n{'Team':<35} {'Boundaries':>10}")
print("-" * 48)
for i in team_top5:
    print(f"{unique_teams[i]:<35} {team_boundaries[i]:>10}")
print()

Total Fours : 29,850
Total Sixes : 13,051

Team with most boundaries : Mumbai Indians (5,322 boundaries)

Team                                Boundaries
------------------------------------------------
Mumbai Indians                            5322
Kolkata Knight Riders                     4956
Chennai Super Kings                       4705
Royal Challengers Bangalore               4637
Rajasthan Royals                          4328



TASK 7 — Death Overs Analysis (overs 16–20)

In [22]:

death_mask   = (over >= 15) & (over <= 19)   # 0-indexed: 15–19 = overs 16–20
death_runs   = batsman_runs[death_mask].sum()
death_teams  = batting_team[death_mask]
death_bruns  = batsman_runs[death_mask]
 
print(f"Total batsman runs in death overs : {death_runs:,}")
 
# Team with highest death-over runs
team_death_runs = np.array(
    [death_bruns[death_teams == t].sum() for t in unique_teams]
)
best_death = np.argmax(team_death_runs)
print(f"Team with highest death-over runs : {unique_teams[best_death]}"
      f" ({team_death_runs[best_death]:,})")
print()

Total batsman runs in death overs : 88,907
Team with highest death-over runs : Mumbai Indians (11,245)



TASK 8 — Highest Scoring Match

In [23]:

best_match_idx = np.argmax(total_runs_per_match)
best_mid       = unique_matches[best_match_idx]
best_runs      = total_runs_per_match[best_match_idx]
print(f"Match ID : {best_mid}   |   Total Batsman Runs : {best_runs}")
print()
 

Match ID : 1426268   |   Total Batsman Runs : 520



TASK 9 — Match Winner Approximation (sample: first 10 matches)

In [24]:

print(f"{'Match ID':>10}  {'Team A':<35} {'Runs A':>7}  {'Team B':<35} {'Runs B':>7}  {'Approx Winner'}")
print("-" * 115)
 
for mid in unique_matches[:10]:
    mask_m   = match_id == mid
    teams_in = np.unique(batting_team[mask_m])
 
    if len(teams_in) < 2:
        continue
 
    r0 = batsman_runs[mask_m & (batting_team == teams_in[0])].sum()
    r1 = batsman_runs[mask_m & (batting_team == teams_in[1])].sum()
    winner = teams_in[0] if r0 > r1 else teams_in[1]
    print(f"{mid:>10}  {teams_in[0]:<35} {r0:>7}  {teams_in[1]:<35} {r1:>7}  {winner}")
print()
 

  Match ID  Team A                               Runs A  Team B                               Runs B  Approx Winner
-------------------------------------------------------------------------------------------------------------------
    335982  Kolkata Knight Riders                   205  Royal Challengers Bangalore              63  Kolkata Knight Riders
    335983  Chennai Super Kings                     234  Kings XI Punjab                         196  Chennai Super Kings
    335984  Delhi Daredevils                        122  Rajasthan Royals                        122  Rajasthan Royals
    335985  Mumbai Indians                          154  Royal Challengers Bangalore             161  Royal Challengers Bangalore
    335986  Deccan Chargers                         100  Kolkata Knight Riders                    84  Deccan Chargers
    335987  Kings XI Punjab                         162  Rajasthan Royals                        156  Kings XI Punjab
    335988  Deccan Chargers          

TASK 10 — Toss Impact Analysis

In [25]:
sort_order     = np.argsort(m_id)
m_id_sorted    = m_id[sort_order]
toss_sorted    = m_toss_winner[sort_order]
 

positions      = np.searchsorted(m_id_sorted, match_id)

positions      = np.clip(positions, 0, len(m_id_sorted) - 1)
 

valid          = m_id_sorted[positions] == match_id
toss_per_del   = toss_sorted[positions]   
 

toss_batting_mask = (batting_team == toss_per_del) & valid
opp_batting_mask  = (batting_team != toss_per_del) & valid
 
toss_runs = batsman_runs[toss_batting_mask].sum()
opp_runs  = batsman_runs[opp_batting_mask].sum()
 
toss_match_runs = np.array(
    [batsman_runs[toss_batting_mask & (match_id == mid)].sum()
     for mid in unique_matches]
)
opp_match_runs = np.array(
    [batsman_runs[opp_batting_mask & (match_id == mid)].sum()
     for mid in unique_matches]
)
 
avg_toss = toss_match_runs.mean()
avg_opp  = opp_match_runs.mean()
 
print(f"Avg runs scored by TOSS WINNER per match : {avg_toss:.2f}")
print(f"Avg runs scored by OPPONENT    per match : {avg_opp:.2f}")
print(f"Toss winner scores more        : {'YES' if avg_toss > avg_opp else 'NO'}")
print()

Avg runs scored by TOSS WINNER per match : 148.38
Avg runs scored by OPPONENT    per match : 153.04
Toss winner scores more        : NO



TASK 11 — Match Scorecards (first 5 matches)

In [28]:

for mid in unique_matches[:5]:
    mask_m    = match_id == mid
    teams_in  = np.unique(batting_team[mask_m])
    print(f"\nMatch {mid}:")
    for team in teams_in:
        team_mask  = mask_m & (batting_team == team)
        runs       = batsman_runs[team_mask].sum()
        extras     = extra_runs[team_mask].sum()
        wickets    = is_wicket[team_mask].sum()
        boundaries_4 = np.sum(batsman_runs[team_mask] == 4)
        boundaries_6 = np.sum(batsman_runs[team_mask] == 6)
        print(f"  {team:<35}  Runs: {runs:>4}  Extras: {extras:>3}"
              f"  Wickets: {wickets:>2}  4s: {boundaries_4:>3}  6s: {boundaries_6:>3}")
 
print()



Match 335982:
  Kolkata Knight Riders                Runs:  205  Extras:  17  Wickets:  3  4s:  15  6s:  14
  Royal Challengers Bangalore          Runs:   63  Extras:  19  Wickets: 10  4s:   3  6s:   3

Match 335983:
  Chennai Super Kings                  Runs:  234  Extras:   6  Wickets:  5  4s:  20  6s:  16
  Kings XI Punjab                      Runs:  196  Extras:  11  Wickets:  4  4s:  18  6s:   9

Match 335984:
  Delhi Daredevils                     Runs:  122  Extras:  10  Wickets:  1  4s:  18  6s:   1
  Rajasthan Royals                     Runs:  122  Extras:   7  Wickets:  8  4s:  14  6s:   3

Match 335985:
  Mumbai Indians                       Runs:  154  Extras:  11  Wickets:  7  4s:  18  6s:   5
  Royal Challengers Bangalore          Runs:  161  Extras:   5  Wickets:  5  4s:  15  6s:   6

Match 335986:
  Deccan Chargers                      Runs:  100  Extras:  10  Wickets: 10  4s:   6  6s:   6
  Kolkata Knight Riders                Runs:   84  Extras:  28  Wickets:  5  4s